# DASYNET: PneumoniaMNIST Classification

This notebook implements **DASYNET**—a custom Convolutional Neural Network (CNN) built from scratch to classify chest X-rays (normal vs. pneumonia) from the MedMNIST collection.

The goal is to match benchmark-level accuracy (typically ~90-95% for simple lightweight CNNs on this dataset) using a logical sequence of Convolution, Batch Normalization, and Max Pooling layers.

In [1]:
# Colab setup for Blackwell GPUs (sm_120)
# After this installs torch, RESTART runtime and run all cells from top.
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install -U medmnist
!pip -q install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128 || !pip -q install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu129

import torch

print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))
    print("If capability is (12, 0), this build must include sm_120 support.")
else:
    print("No CUDA GPU detected by runtime. In Colab, set Runtime > Change runtime type > GPU.")

print("If you still see an sm_120 compatibility warning, restart runtime once and rerun this cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 40.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 99.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 28.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 12.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO, Evaluator
from torch.utils.data import DataLoader

print(f"MedMNIST v{medmnist.__version__} @ {medmnist.__path__}")


def resolve_device() -> torch.device:
    """Use CUDA only if a tiny CUDA op succeeds; otherwise fallback to CPU."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _ = torch.randn(1, device="cuda")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA unavailable at runtime ({e}). Falling back to CPU.")
        return torch.device("cpu")


# Check device compatibility
device = resolve_device()
print(f"Using device: {device}")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

# 2. Data Loading and Preprocessing
data_flag = 'pneumoniamnist'
download = True
IMG_SIZE = 224

# Tuned for G4/T4-class GPU at 224.
BATCH_SIZE = 192 if device.type == 'cuda' else 32

info = INFO[data_flag]
n_classes = len(info['label'])
DataClass = getattr(medmnist, info['python_class'])

# Define transformations for 224x224 grayscale inputs
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load datasets at 224x224
train_dataset = DataClass(split='train', transform=data_transform, download=download, size=IMG_SIZE)
val_dataset = DataClass(split='val', transform=data_transform, download=download, size=IMG_SIZE)
test_dataset = DataClass(split='test', transform=data_transform, download=download, size=IMG_SIZE)

if device.type == 'cuda':
    loader_kwargs = {
        "num_workers": 4,
        "pin_memory": True,
        "persistent_workers": True,
        "prefetch_factor": 2
    }
else:
    loader_kwargs = {
        "num_workers": 2,
        "pin_memory": False
    }

# Create Data Loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
# Use non-shuffled loaders for metric computation with MedMNIST evaluator
train_eval_loader = DataLoader(dataset=train_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)
val_loader = DataLoader(dataset=val_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(dataset=test_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"Training set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")
print(f"Batch size: {BATCH_SIZE}")

# 3. DASYNET Architecture
class DASYNET(nn.Module):
    def __init__(self, in_channels=1, num_classes=n_classes):
        super(DASYNET, self).__init__()

        # Block 1 - "Scanner": 224x224 -> 112x112
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2 - "Scanner": 112x112 -> 56x56
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3 - "Scanner": 56x56 -> 28x28
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Keep classifier dimensions stable across input sizes.
        self.global_pool = nn.AdaptiveAvgPool2d((3, 3))
        self.flatten = nn.Flatten()

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(64 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

model = DASYNET(in_channels=1, num_classes=n_classes).to(device)
if device.type == 'cuda':
    model = model.to(memory_format=torch.channels_last)

# 4. Initialization, Optimizer, and Loss
EPOCHS = 35
LEARNING_RATE = 1e-3

# Class-balanced + label-smoothed CE to improve generalization at 224.
train_labels = torch.as_tensor(train_dataset.labels).squeeze().long()
class_counts = torch.bincount(train_labels, minlength=n_classes).float()
class_weights = (class_counts.sum() / (n_classes * class_counts.clamp_min(1.0))).to(device)
print(f"Class counts: {class_counts.tolist()}")
print(f"Class weights: {[round(float(x), 4) for x in class_weights]}")

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# 5. Model Training Loop (best-checkpoint + early stopping)
best_val_loss = float('inf')
best_epoch = 0
best_state = copy.deepcopy(model.state_dict())
patience = 8
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).squeeze().long()

        if device.type == 'cuda':
            inputs = inputs.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        if device.type == 'cuda':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += outputs.max(1)[1].eq(targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True).squeeze().long()

            if device.type == 'cuda':
                inputs = inputs.contiguous(memory_format=torch.channels_last)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_correct += outputs.max(1)[1].eq(targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = 100. * val_correct / val_total
    scheduler.step(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}; best epoch was {best_epoch} (val loss {best_val_loss:.4f}).")
        break

# Always evaluate best validation checkpoint.
model.load_state_dict(best_state)
print(f"Loaded best checkpoint from epoch {best_epoch} (val loss {best_val_loss:.4f}).")

# 6. Evaluation
def test(split):
    model.eval()
    y_score = torch.tensor([])

    if split == 'train':
        loader = train_eval_loader
    elif split == 'test':
        loader = test_loader
    else:
        raise ValueError("split must be 'train' or 'test'")

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device, non_blocking=True)
            if device.type == 'cuda':
                inputs = inputs.contiguous(memory_format=torch.channels_last)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                scores = F.softmax(outputs, dim=1)

            y_score = torch.cat((y_score, scores.cpu()), 0)

    metrics = medmnist.Evaluator(data_flag, split, size=IMG_SIZE).evaluate(y_score.numpy())
    print(f'{split.capitalize()} Set AUC: {metrics[0]:.4f} | Accuracy: {metrics[1]:.4f}')

print("\n--- Final Benchmarking ---")
test('train')
test('test')

MedMNIST v3.0.2 @ ['/usr/local/lib/python3.12/dist-packages/medmnist']
Using device: cuda


100%|██████████| 214M/214M [00:01<00:00, 153MB/s]  


Training set: 4708 images
Test set: 624 images
Batch size: 192
Class counts: [1214.0, 3494.0]
Class weights: [1.939, 0.6737]
Epoch [1/35] Train Loss: 0.3986, Acc: 85.07% | Val Loss: 1.8127, Acc: 74.24%
Epoch [2/35] Train Loss: 0.3058, Acc: 91.72% | Val Loss: 1.2448, Acc: 74.24%
Epoch [3/35] Train Loss: 0.2838, Acc: 93.29% | Val Loss: 0.3466, Acc: 94.66%
Epoch [4/35] Train Loss: 0.2677, Acc: 93.97% | Val Loss: 0.2633, Acc: 94.66%
Epoch [5/35] Train Loss: 0.2656, Acc: 94.50% | Val Loss: 0.3432, Acc: 83.78%
Epoch [6/35] Train Loss: 0.2674, Acc: 94.14% | Val Loss: 0.2763, Acc: 91.22%
Epoch [7/35] Train Loss: 0.2543, Acc: 95.35% | Val Loss: 0.3110, Acc: 87.02%
Epoch [8/35] Train Loss: 0.2509, Acc: 95.41% | Val Loss: 0.2637, Acc: 93.32%
Epoch [9/35] Train Loss: 0.2407, Acc: 95.52% | Val Loss: 0.2403, Acc: 96.18%
Epoch [10/35] Train Loss: 0.2358, Acc: 95.86% | Val Loss: 0.2401, Acc: 95.99%
Epoch [11/35] Train Loss: 0.2312, Acc: 96.22% | Val Loss: 0.2571, Acc: 93.70%
Epoch [12/35] Train Loss: 

In [3]:
# Save trained weights with robust Colab fallbacks
import os
import shutil
import torch

model_path = 'dasynet_pneumonia.pth'
torch.save(model.state_dict(), model_path)

abs_path = os.path.abspath(model_path)
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Saved model to: {abs_path}")
print(f"File size: {size_mb:.2f} MB")

# Try browser download (works in standard Colab browser sessions).
try:
    from google.colab import files
    files.download(model_path)
    print('Download trigger sent. If nothing appears, use the Drive copy path below.')
except Exception as e:
    print(f'Browser download not available in this client: {e}')

# Always create a persistent Drive copy when available.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_dir = '/content/drive/MyDrive/IDS705'
    os.makedirs(drive_dir, exist_ok=True)
    drive_path = os.path.join(drive_dir, model_path)
    shutil.copy2(model_path, drive_path)
    print(f'Drive backup saved to: {drive_path}')
except Exception as e:
    print(f'Drive backup skipped: {e}')

print('Done. If browser download did not start, get the file from MyDrive/IDS705.')

Mounted at /content/drive
Drive backup saved to: /content/drive/MyDrive/IDS705/dasynet_pneumonia.pth
Done. If browser download did not start, get the file from MyDrive/IDS705.
